# L11 · Training ACT and SmolVLA Policies

This notebook turns the lecture's evidence ladder into an executable workflow: validate one real sample, inspect installed policy defaults, audit both generated commands, optionally run one GPU step per policy, inspect the resulting checkpoints, and reload both policies on the same sample.

The complete lab requires a GPU. The default path keeps both smoke runs disabled so that opening or running the notebook never starts expensive work without an explicit choice. A dry-run proves command assembly only; no result in this notebook is closed-loop task evidence.

## Before you run

Install the complete course environment and, on AMD, replace the portable PyTorch packages with the verified ROCm wheels described in COMPATIBILITY.md. Start Jupyter from that environment without adding paths to Python.

Set these environment variables before starting the kernel:

- RG101_REPO_ID: logical LeRobot dataset ID; defaults to genesis/fruit_pick.
- RG101_DATASET_ROOT: local dataset directory; defaults to datasets/fruit_pick.
- RG101_OUTPUT_ROOT: generated run directory; defaults to outputs/train/l11.
- RG101_SEED: recorded seed; defaults to 1000.
- RG101_RUN_SMOKE: leave at 0 for data and command checks; set to 1 only for the two real GPU smoke runs.
- RG101_SMOLVLA_BASE_SNAPSHOT and RG101_SMOLVLA_VLM_SNAPSHOT: exact local Hugging Face snapshot directories for the verified commits. Set both or neither.

Select the intended GPU before launching the kernel. On the verified AMD system, PyTorch exposes ROCm through the torch.cuda API, so the LeRobot device string remains cuda. CPU fallback does not count as a smoke pass.

In [ ]:
import importlib.metadata as package_metadata
import json
import os
import shlex
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import torch
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.paths import DATASETS_DIR, PATHS, TRAIN_OUTPUTS_DIR
from robo_genesis.training_contract import (
    SMOLVLA_BASE_REPO_ID,
    SMOLVLA_BASE_REVISION,
    SMOLVLA_CAMERA_RENAME,
    SMOLVLA_VLM_REPO_ID,
    SMOLVLA_VLM_REVISION,
    audit_checkpoint,
    audit_smolvla_snapshots,
    command_options,
    horizon_evidence,
    parse_training_metrics,
    resolve_numeric_checkpoint,
)


def environment_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    normalized = value.strip().lower()
    if normalized not in {"0", "1", "false", "true", "no", "yes"}:
        raise ValueError(f"{name} must be a boolean value, got {value!r}")
    return normalized in {"1", "true", "yes"}


def installed_version(distribution):
    try:
        return package_metadata.version(distribution)
    except package_metadata.PackageNotFoundError:
        return "not-installed"


lesson = load_course_manifest().lesson("L11")
assert lesson.status.value == "gpu-verified"

repo_id = os.environ.get("RG101_REPO_ID", "genesis/fruit_pick")
dataset_root = Path(
    os.environ.get("RG101_DATASET_ROOT", str(DATASETS_DIR / "fruit_pick"))
).expanduser().resolve()
output_root = Path(
    os.environ.get("RG101_OUTPUT_ROOT", str(TRAIN_OUTPUTS_DIR / "l11"))
).expanduser().resolve()
seed = int(os.environ.get("RG101_SEED", "1000"))
run_smoke = environment_flag("RG101_RUN_SMOKE", default=False)

base_snapshot_value = os.environ.get("RG101_SMOLVLA_BASE_SNAPSHOT")
vlm_snapshot_value = os.environ.get("RG101_SMOLVLA_VLM_SNAPSHOT")
base_snapshot = Path(base_snapshot_value).expanduser().resolve() if base_snapshot_value else None
vlm_snapshot = Path(vlm_snapshot_value).expanduser().resolve() if vlm_snapshot_value else None

output_root.mkdir(parents=True, exist_ok=True)
cuda_available = torch.cuda.is_available()
visible_devices = torch.cuda.device_count()
actual_device = torch.cuda.get_device_name(0) if cuda_available else None
runtime_evidence = {
    "python": sys.version.split()[0],
    "lerobot": installed_version("lerobot"),
    "torch": torch.__version__,
    "torch_hip": torch.version.hip,
    "torch_cuda": torch.version.cuda,
    "cuda_api_available": cuda_available,
    "visible_devices": visible_devices,
    "actual_device": actual_device,
    "repo_id": repo_id,
    "dataset_root": str(dataset_root),
    "output_root": str(output_root),
    "seed": seed,
    "run_smoke": run_smoke,
    "hf_hub_offline": os.environ.get("HF_HUB_OFFLINE"),
    "transformers_offline": os.environ.get("TRANSFORMERS_OFFLINE"),
}
print(json.dumps(runtime_evidence, indent=2, ensure_ascii=False))

if run_smoke and not cuda_available:
    raise RuntimeError(
        "RG101_RUN_SMOKE=1 requires an actual GPU. CPU fallback is not accepted for this lab."
    )

## 1. Fail early at the data boundary

The first executable gate reads metadata and one real decoded sample before allocating either model. It checks the shared 9-D state/action joint order, both RGB views, task text, FPS, and episode/frame counts. A missing or malformed dataset raises an actionable error instead of being replaced by random tensors.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

from robo_genesis.record_dataset import JOINT_NAMES

info_path = dataset_root / "meta" / "info.json"
if not info_path.is_file():
    raise FileNotFoundError(
        f"Dataset metadata is missing: {info_path}. "
        "Set RG101_DATASET_ROOT to the local LeRobot dataset produced in the earlier lessons."
    )

metadata = LeRobotDatasetMetadata(repo_id, root=dataset_root)
dataset = LeRobotDataset(repo_id, root=dataset_root, video_backend="pyav")
if len(dataset) < 1:
    raise ValueError(f"Dataset contains no readable samples: {dataset_root}")
sample = dataset[0]

STATE_KEY = "observation.state"
ACTION_KEY = "action"
IMAGE_KEYS = (
    "observation.images.world",
    "observation.images.wrist",
)


def as_numpy(value):
    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    return np.asarray(value)


def check_vector(key):
    feature = metadata.features.get(key)
    if feature is None:
        raise KeyError(f"Dataset metadata is missing {key}")
    if tuple(feature["shape"]) != (9,) or feature["dtype"] != "float32":
        raise ValueError(f"{key} metadata must be shape (9,) float32, got {feature}")
    if tuple(feature.get("names") or ()) != tuple(JOINT_NAMES):
        raise ValueError(f"{key} joint names do not match the course joint order")

    value = as_numpy(sample[key])
    if value.shape != (9,) or value.dtype != np.float32:
        raise ValueError(f"{key} sample must be shape (9,) float32, got {value.shape} {value.dtype}")
    if not np.isfinite(value).all():
        raise ValueError(f"{key} sample contains non-finite values")
    return value


state = check_vector(STATE_KEY)
action = check_vector(ACTION_KEY)
decoded_shapes = {}
for key in IMAGE_KEYS:
    feature = metadata.features.get(key)
    if feature is None:
        raise KeyError(f"Dataset metadata is missing {key}")
    stored_h, stored_w, stored_c = tuple(feature["shape"])
    if stored_c != 3:
        raise ValueError(f"{key} metadata must describe RGB HWC frames, got {feature['shape']}")

    image = as_numpy(sample[key])
    if image.shape != (3, stored_h, stored_w):
        raise ValueError(
            f"{key} decoded sample must be CHW {(3, stored_h, stored_w)}, got {image.shape}"
        )
    if not np.isfinite(image).all():
        raise ValueError(f"{key} decoded sample contains non-finite values")
    decoded_shapes[key] = tuple(image.shape)

if decoded_shapes[IMAGE_KEYS[0]] != decoded_shapes[IMAGE_KEYS[1]]:
    raise ValueError(f"Camera shapes differ: {decoded_shapes}")

task_text = sample.get("task")
if not isinstance(task_text, str) or not task_text.strip():
    raise ValueError("The real dataset sample must resolve to non-empty task text")
if metadata.fps <= 0 or metadata.total_episodes <= 0 or metadata.total_frames <= 0:
    raise ValueError("Dataset FPS, episode count, and frame count must all be positive")

dataset_evidence = {
    "status": "PASS",
    "repo_id": repo_id,
    "root": str(metadata.root),
    "episodes": metadata.total_episodes,
    "frames": metadata.total_frames,
    "fps": metadata.fps,
    "state": {"shape": state.shape, "dtype": str(state.dtype), "finite": True},
    "action": {"shape": action.shape, "dtype": str(action.dtype), "finite": True},
    "decoded_images": decoded_shapes,
    "task": task_text,
}
print(json.dumps(dataset_evidence, indent=2, ensure_ascii=False, default=list))

## 2. Connect installed configuration to time

Read policy defaults from the installed LeRobot version rather than copying values into the notebook. The prediction horizon is chunk_size divided by dataset FPS; the replanning interval is n_action_steps divided by FPS. These are different even though both defaults currently use equal values.

The reduced ACT entry is explicitly a pipeline-only smoke configuration. It is not a small baseline for comparing task quality with SmolVLA.

In [ ]:
act_default = ACTConfig()
smolvla_default = SmolVLAConfig()

configuration_evidence = {
    "act_default": {
        "horizon": horizon_evidence(
            chunk_size=act_default.chunk_size,
            n_action_steps=act_default.n_action_steps,
            fps=metadata.fps,
        ),
        "pretrained_backbone_weights": act_default.pretrained_backbone_weights,
    },
    "act_pipeline_only_smoke": horizon_evidence(
        chunk_size=10,
        n_action_steps=10,
        fps=metadata.fps,
    ),
    "smolvla_default": {
        "horizon": horizon_evidence(
            chunk_size=smolvla_default.chunk_size,
            n_action_steps=smolvla_default.n_action_steps,
            fps=metadata.fps,
        ),
        "vlm_model_name": smolvla_default.vlm_model_name,
        "freeze_vision_encoder": smolvla_default.freeze_vision_encoder,
        "train_expert_only": smolvla_default.train_expert_only,
        "train_state_proj": smolvla_default.train_state_proj,
    },
}
print(json.dumps(configuration_evidence, indent=2, ensure_ascii=False, default=str))

## 3. Audit both SmolVLA model revisions

A local base snapshot alone does not pin the VLM loaded by SmolVLA. The gate therefore checks both exact 40-character revisions, required files, the base policy type, and the VLM repository named by its configuration.

An exact Hugging Face cache path proves its revision through the snapshots/commit directory. A copied snapshot may instead carry a robo_genesis_snapshot.json provenance file with repo_id and revision. The audit reads local files only; it never downloads a model.

In [ ]:
if (base_snapshot is None) != (vlm_snapshot is None):
    raise ValueError(
        "Set both RG101_SMOLVLA_BASE_SNAPSHOT and RG101_SMOLVLA_VLM_SNAPSHOT, or neither."
    )

snapshot_evidence = None
if base_snapshot is not None and vlm_snapshot is not None:
    snapshot_evidence = audit_smolvla_snapshots(base_snapshot, vlm_snapshot)
    print("PASS — pinned SmolVLA base and VLM snapshots")
    print(json.dumps(snapshot_evidence.as_dict(), indent=2, ensure_ascii=False))
else:
    print("NOT READY — pinned SmolVLA model content was not supplied")
    print(f"expected base: {SMOLVLA_BASE_REPO_ID}@{SMOLVLA_BASE_REVISION}")
    print(f"expected VLM:  {SMOLVLA_VLM_REPO_ID}@{SMOLVLA_VLM_REVISION}")

if run_smoke and snapshot_evidence is None:
    raise RuntimeError(
        "RG101_RUN_SMOKE=1 requires both audited local SmolVLA snapshot paths."
    )

## 4. Generate and audit both dry-run commands

These calls execute the project wrapper with --dry-run. They do not open the dataset, load either model, allocate a GPU tensor, or write a checkpoint. The code parses the wrapper's shell-safe output and checks paths, job names, default batches, device, PyAV, disabled publishing, policy selection, and the SmolVLA camera map.

When pinned snapshots were supplied, the SmolVLA command also carries the audited local VLM path. Without them, the wrapper clearly reports UNPINNED; that command remains useful for discovery but is not ready for a reproducible training run.

In [ ]:
def run_wrapper(command):
    print("$ " + shlex.join(command))
    started = time.perf_counter()
    completed = subprocess.run(
        command,
        cwd=PATHS.project_root,
        text=True,
        capture_output=True,
        check=False,
    )
    elapsed = time.perf_counter() - started
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with return code {completed.returncode}")
    return {"completed": completed, "elapsed_seconds": elapsed}


def printed_trainer_command(stdout):
    lines = [line for line in stdout.splitlines() if line.startswith("[train] ")]
    if len(lines) != 1:
        raise ValueError(f"Expected one [train] command line, found {len(lines)}")
    return shlex.split(lines[0].removeprefix("[train] "))


def wrapper_command(policy, name, *, batch_size=None, dry_run=False):
    command = [
        sys.executable,
        "-m",
        "robo_genesis.train_policy",
        policy,
        "--repo-id",
        repo_id,
        "--dataset-root",
        str(dataset_root),
        "--name",
        name,
        "--output-dir",
        str(output_root / name),
        "--steps",
        "1",
        "--save-freq",
        "1",
        "--log-freq",
        "1",
        "--num-workers",
        "0",
        "--seed",
        str(seed),
        "--device",
        "cuda",
        "--video-backend",
        "pyav",
    ]
    if batch_size is not None:
        command.extend(["--batch-size", str(batch_size)])
    if policy == "smolvla" and snapshot_evidence is not None:
        command.extend(
            [
                "--policy-path",
                str(snapshot_evidence.base.path),
                "--smolvla-vlm-path",
                str(snapshot_evidence.vlm.path),
            ]
        )
    if dry_run:
        command.append("--dry-run")
    return command


def check_common_options(options, name, expected_batch):
    expected = {
        "--dataset.repo_id": repo_id,
        "--dataset.root": str(dataset_root),
        "--output_dir": str((output_root / name).resolve()),
        "--job_name": name,
        "--batch_size": str(expected_batch),
        "--steps": "1",
        "--save_freq": "1",
        "--log_freq": "1",
        "--num_workers": "0",
        "--seed": str(seed),
        "--policy.device": "cuda",
        "--policy.push_to_hub": "false",
        "--wandb.enable": "false",
        "--dataset.video_backend": "pyav",
    }
    for option, expected_value in expected.items():
        if options.get(option) != expected_value:
            raise AssertionError(f"{option}: {options.get(option)!r} != {expected_value!r}")


act_dry_name = "l11-act-dry-run"
act_dry_result = run_wrapper(wrapper_command("act", act_dry_name, dry_run=True))
act_trainer_command = printed_trainer_command(act_dry_result["completed"].stdout)
act_options = command_options(act_trainer_command)
check_common_options(act_options, act_dry_name, expected_batch=8)
assert act_options["--policy.type"] == "act"
assert "--policy.path" not in act_options
assert "--rename_map" not in act_options

smolvla_dry_name = "l11-smolvla-dry-run"
smolvla_dry_result = run_wrapper(wrapper_command("smolvla", smolvla_dry_name, dry_run=True))
smolvla_trainer_command = printed_trainer_command(smolvla_dry_result["completed"].stdout)
smolvla_options = command_options(smolvla_trainer_command)
check_common_options(smolvla_options, smolvla_dry_name, expected_batch=4)
expected_smolvla_path = (
    str(snapshot_evidence.base.path)
    if snapshot_evidence is not None
    else SMOLVLA_BASE_REPO_ID
)
assert smolvla_options["--policy.path"] == expected_smolvla_path
assert json.loads(str(smolvla_options["--rename_map"])) == SMOLVLA_CAMERA_RENAME
if snapshot_evidence is None:
    assert "--policy.vlm_model_name" not in smolvla_options
else:
    assert smolvla_options["--policy.vlm_model_name"] == str(snapshot_evidence.vlm.path)

print("PASS — ACT and SmolVLA command contracts; no dataset, model, or GPU was opened")

## 5. Prepare the two one-step GPU smokes

The ACT smoke disables ImageNet downloads and reduces the CVAE/Transformer only to exercise decode, forward, loss, backward, update, and save. SmolVLA retains its verified base architecture and fine-tuning boundary while using both pinned local snapshots.

The preflight below is still a dry-run. The following cell starts training only when RG101_RUN_SMOKE=1 was set before the kernel started.

In [ ]:
act_smoke_command = wrapper_command("act", "l11-act-smoke", batch_size=1)
act_smoke_command.extend(
    [
        "--",
        "--policy.pretrained_backbone_weights=null",
        "--policy.chunk_size=10",
        "--policy.n_action_steps=10",
        "--policy.dim_model=64",
        "--policy.n_heads=4",
        "--policy.dim_feedforward=128",
        "--policy.n_encoder_layers=1",
        "--policy.n_decoder_layers=1",
        "--policy.n_vae_encoder_layers=1",
        "--policy.latent_dim=8",
    ]
)

smolvla_smoke_command = (
    wrapper_command("smolvla", "l11-smolvla-smoke", batch_size=1)
    if snapshot_evidence is not None
    else None
)


def as_dry_run(command):
    result = list(command)
    separator = result.index("--") if "--" in result else len(result)
    result.insert(separator, "--dry-run")
    return result


act_preflight = run_wrapper(as_dry_run(act_smoke_command))
act_smoke_options = command_options(
    printed_trainer_command(act_preflight["completed"].stdout)
)
assert act_smoke_options["--policy.pretrained_backbone_weights"] == "null"
assert act_smoke_options["--policy.chunk_size"] == "10"
assert act_smoke_options["--policy.n_action_steps"] == "10"
assert act_smoke_options["--policy.dim_model"] == "64"
print("PASS — ACT command is a pipeline-only smoke configuration")

if smolvla_smoke_command is None:
    print("SKIP — SmolVLA smoke preflight requires the two pinned local snapshots")
else:
    smolvla_preflight = run_wrapper(as_dry_run(smolvla_smoke_command))
    smolvla_smoke_options = command_options(
        printed_trainer_command(smolvla_preflight["completed"].stdout)
    )
    assert smolvla_smoke_options["--policy.path"] == str(snapshot_evidence.base.path)
    assert smolvla_smoke_options["--policy.vlm_model_name"] == str(snapshot_evidence.vlm.path)
    assert json.loads(str(smolvla_smoke_options["--rename_map"])) == SMOLVLA_CAMERA_RENAME
    print("PASS — SmolVLA smoke command uses both pinned snapshots and the camera map")

In [ ]:
smoke_processes = {}
if run_smoke:
    assert smolvla_smoke_command is not None
    smoke_processes["act"] = run_wrapper(act_smoke_command)
    smoke_processes["smolvla"] = run_wrapper(smolvla_smoke_command)
else:
    print("SKIP — GPU smoke runs are opt-in; set RG101_RUN_SMOKE=1 before starting the kernel")

## 6. Inspect finite metrics and checkpoint packages

A passing subprocess is not enough. For each executed smoke, require a log record containing finite loss and gradient norm, verify that checkpoints/last resolves to the newest numeric step, and inspect the non-empty weights, policy configuration, training configuration, pre/postprocessors, and referenced processor state files.

The two loss values have different objectives and scales. Do not compare them as a policy ranking, and do not draw a trend from one point.

In [ ]:
checkpoint_evidence = {}
if run_smoke:
    for policy_name, process in smoke_processes.items():
        combined_log = process["completed"].stdout + "\n" + process["completed"].stderr
        metrics = parse_training_metrics(combined_log)
        run_directory = output_root / f"l11-{policy_name}-smoke"
        model_directory = resolve_numeric_checkpoint(run_directory)
        artifact = audit_checkpoint(
            model_directory,
            expected_policy_type=policy_name,
            expected_action_dim=9,
        )
        if artifact["dataset_repo_id"] != repo_id:
            raise AssertionError(f"{policy_name} checkpoint records the wrong dataset")
        if artifact["seed"] != seed:
            raise AssertionError(f"{policy_name} checkpoint records the wrong seed")
        artifact["metrics"] = metrics
        artifact["elapsed_seconds_this_run"] = process["elapsed_seconds"]
        checkpoint_evidence[policy_name] = artifact
    print(json.dumps(checkpoint_evidence, indent=2, ensure_ascii=False, default=str))
else:
    print("SKIP — no checkpoint is claimed because GPU smoke was not run")

## 7. Reload both checkpoints on the same real sample

LeRobot decodes video samples as channel-first floating tensors, while the project inference helper consumes raw HWC uint8 images before applying the saved preprocessor. The conversion below is explicit and checked.

Each policy must return one finite 9-D float32 action. This is an open-loop single-sample probe: Genesis is not constructed, the action is not applied, and task success is not evaluated.

In [ ]:
def decoded_rgb_to_hwc_uint8(value):
    image = as_numpy(value)
    if image.ndim != 3:
        raise ValueError(f"Decoded image must have three dimensions, got {image.shape}")
    if image.shape[0] in (3, 4):
        image = np.moveaxis(image, 0, -1)
    if image.shape[-1] == 4:
        image = image[..., :3]
    if image.shape[-1] != 3 or not np.isfinite(image).all():
        raise ValueError(f"Decoded image is not finite RGB data: {image.shape}")
    if np.issubdtype(image.dtype, np.floating):
        if image.min() < 0.0 or image.max() > 1.0 + 1e-6:
            raise ValueError("Floating decoded images must be in the [0, 1] range")
        image = np.rint(np.clip(image, 0.0, 1.0) * 255.0).astype(np.uint8)
    elif image.dtype != np.uint8:
        raise ValueError(f"Unsupported decoded image dtype: {image.dtype}")
    return np.ascontiguousarray(image)


raw_observation = {
    STATE_KEY: np.ascontiguousarray(state, dtype=np.float32),
    **{key: decoded_rgb_to_hwc_uint8(sample[key]) for key in IMAGE_KEYS},
}
assert all(raw_observation[key].shape[-1] == 3 for key in IMAGE_KEYS)

reload_evidence = {}
if run_smoke:
    import gc

    from robo_genesis.eval_policy import load_policy

    for policy_name in ("act", "smolvla"):
        bundle = load_policy(
            checkpoint_evidence[policy_name]["path"],
            repo_id,
            str(dataset_root),
            "cuda",
        )
        bundle.reset()
        predicted_action = bundle.select_action(raw_observation, task_text)
        if predicted_action.shape != (9,):
            raise AssertionError(f"{policy_name} action shape is {predicted_action.shape}, not (9,)")
        if predicted_action.dtype != np.float32:
            raise AssertionError(f"{policy_name} action dtype is {predicted_action.dtype}, not float32")
        if not np.isfinite(predicted_action).all():
            raise AssertionError(f"{policy_name} action contains non-finite values")
        reload_evidence[policy_name] = {
            "policy_type": bundle.policy_type,
            "actual_device": str(bundle.device),
            "raw_image_keys": list(IMAGE_KEYS),
            "task_text_present": bool(task_text.strip()),
            "action_shape": predicted_action.shape,
            "action_dtype": str(predicted_action.dtype),
            "action_finite": True,
        }
        del bundle
        gc.collect()
        torch.cuda.empty_cache()
    print(json.dumps(reload_evidence, indent=2, ensure_ascii=False, default=list))
else:
    print("SKIP — open-loop checkpoint reload requires the two completed smoke runs")

## 8. Design full runs without launching them

Long training is an opt-in take-home experiment. Replace every CHOOSE value only after recording the dataset version and size, model revisions, horizons in seconds, optimization budget, resources, save/log cadence, and the L12 evaluation handoff.

The templates are deliberately invalid until completed. Running this cell prints them and an explicit skip; it never launches either job.

In [ ]:
base_for_template = (
    str(snapshot_evidence.base.path)
    if snapshot_evidence is not None
    else "PINNED_SMOLVLA_BASE_SNAPSHOT"
)
vlm_for_template = (
    str(snapshot_evidence.vlm.path)
    if snapshot_evidence is not None
    else "PINNED_SMOLVLA_VLM_SNAPSHOT"
)

act_full_command = [
    sys.executable,
    "-m",
    "robo_genesis.train_policy",
    "act",
    "--repo-id",
    repo_id,
    "--dataset-root",
    str(dataset_root),
    "--name",
    "act-fruit-pick",
    "--output-dir",
    str(output_root / "act-fruit-pick"),
    "--steps",
    "CHOOSE_INTEGER",
    "--batch-size",
    "CHOOSE_INTEGER",
    "--save-freq",
    "CHOOSE_INTEGER",
    "--log-freq",
    "CHOOSE_INTEGER",
    "--num-workers",
    "CHOOSE_INTEGER",
    "--seed",
    "CHOOSE_INTEGER",
    "--device",
    "cuda",
    "--video-backend",
    "pyav",
    "--",
    "--policy.pretrained_backbone_weights=CHOOSE_IMAGENET_IDENTIFIER_OR_NULL",
    "--policy.chunk_size=CHOOSE_INTEGER",
    "--policy.n_action_steps=CHOOSE_INTEGER",
    "--policy.optimizer_lr=CHOOSE_FLOAT",
]

smolvla_full_command = [
    sys.executable,
    "-m",
    "robo_genesis.train_policy",
    "smolvla",
    "--repo-id",
    repo_id,
    "--dataset-root",
    str(dataset_root),
    "--policy-path",
    base_for_template,
    "--smolvla-vlm-path",
    vlm_for_template,
    "--name",
    "smolvla-fruit-pick",
    "--output-dir",
    str(output_root / "smolvla-fruit-pick"),
    "--steps",
    "CHOOSE_INTEGER",
    "--batch-size",
    "CHOOSE_INTEGER",
    "--save-freq",
    "CHOOSE_INTEGER",
    "--log-freq",
    "CHOOSE_INTEGER",
    "--num-workers",
    "CHOOSE_INTEGER",
    "--seed",
    "CHOOSE_INTEGER",
    "--device",
    "cuda",
    "--video-backend",
    "pyav",
    "--",
    "--policy.chunk_size=CHOOSE_INTEGER",
    "--policy.n_action_steps=CHOOSE_INTEGER",
    "--policy.optimizer_lr=CHOOSE_FLOAT",
    "--policy.freeze_vision_encoder=CHOOSE_TRUE_OR_FALSE",
    "--policy.train_expert_only=CHOOSE_TRUE_OR_FALSE",
    "--policy.train_state_proj=CHOOSE_TRUE_OR_FALSE",
]

print("SKIP: full training is opt-in; replace every CHOOSE value from a reviewed run record")
print("ACT template:")
print(shlex.join(act_full_command))
print("SmolVLA template:")
print(shlex.join(smolvla_full_command))

## 9. Evidence reflection

Answer these questions using the records produced above:

1. Which checks passed before any policy was allocated?
2. If the two dry-runs passed but RG101_RUN_SMOKE stayed disabled, what remains untested?
3. If a one-step loss and gradient norm are finite, why is convergence still unknown?
4. Which checkpoint files preserve data identity, policy configuration, processors, and the SmolVLA camera rename?
5. Why does one finite open-loop action still say nothing about grasp success?
6. Which exact artifact and protocol must be handed to L12?

Write NOT RUN for work you did not execute. Do not fill gaps with remembered values from another machine or an earlier compatibility run.

In [ ]:
evidence_summary = {
    "dataset_gate": "PASS — one real sample and metadata checked",
    "runtime_config": "PASS — policy defaults read from installed LeRobot",
    "dry_run": "PASS — command assembly only",
    "smolvla_revisions": (
        "PASS — both local snapshots audited"
        if snapshot_evidence is not None
        else "NOT READY — no pinned local snapshots supplied"
    ),
    "gpu_smoke": (
        "PASS — ACT and SmolVLA each completed one step"
        if run_smoke
        else "NOT RUN — RG101_RUN_SMOKE was not enabled"
    ),
    "checkpoint_reload": (
        "PASS — two open-loop single-sample probes"
        if run_smoke
        else "NOT RUN — requires completed smoke checkpoints"
    ),
    "full_training": "NOT RUN — opt-in take-home experiment",
    "closed_loop_evaluation": "NOT RUN — belongs to L12",
}
print(json.dumps(evidence_summary, indent=2, ensure_ascii=False))

## Connection to L12

L11 ends with versioned, inspectable checkpoint artifacts and a precise account of what was and was not run. L12 will place one such artifact inside the Genesis control loop, execute actions over time, apply seeded task predicates, and report closed-loop outcomes. Do not promote the evidence in this notebook across that boundary.